# ColdStart Killer — Build 3K MVP Dataset from Amazon Reviews 2023

⚠️ Run this notebook only when you are ready to stream/download the selected Amazon Reviews 2023 metadata files.

This notebook:
1. Loads selected Amazon Reviews 2023 metadata.
2. Normalizes product fields.
3. Preserves product images.
4. Builds a high-quality, category-diverse 3,000-item MVP dataset.
5. Saves CSV and reports into `analysis/`.

## 1. Setup imports and paths

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.dataset_loading import load_target_metadata
from src.normalize_amazon import normalize_record
from src.mvp_selection import (
    add_scoring_columns,
    build_quality_control_subset,
    build_diverse_subset,
    category_summary,
    write_mvp_outputs,
)
from src.validation import validate_mvp_dataframe

## 2. Load metadata

In [ ]:
records, load_report = load_target_metadata()
load_report

## 3. Normalize metadata

In [ ]:
normalized_rows = [
    normalize_record(record, category_key=record.get("_source_category", "unknown"))
    for record in records
]
df = pd.DataFrame(normalized_rows)
df = add_scoring_columns(df)
df.shape

## 4. Show field coverage

In [ ]:
coverage = {
    "title": df["title_present"].mean(),
    "store": df["store_present"].mean(),
    "categories": df["categories_present"].mean(),
    "price": df["price_usd"].notna().mean(),
    "details": df["details_present"].mean(),
}
pd.Series(coverage).sort_values(ascending=False)

## 5. Show description vs combined text coverage

In [ ]:
df[["description_words", "features_words", "details_words", "combined_words"]].describe()

## 6. Show image coverage

In [ ]:
image_coverage = {
    "has_images": df["has_images"].mean(),
    "primary_image_url": df["primary_image_url"].notna().mean(),
    "avg_images_count": df["images_count"].mean(),
}
image_coverage

## 7. Build quality-control 3k dataset

In [ ]:
control_df = build_quality_control_subset(df, target_n=3000)
{
    "rows": len(control_df),
    "unique_parent_asin": control_df["parent_asin"].nunique(),
    "selected_threshold": control_df.attrs.get("selected_threshold"),
    "warnings": control_df.attrs.get("warnings", []),
}

## 8. Build diverse 3k dataset

In [ ]:
diverse_df = build_diverse_subset(df, target_n=3000, min_combined_words=150, max_per_category=1200)
{
    "rows": len(diverse_df),
    "unique_parent_asin": diverse_df["parent_asin"].nunique(),
    "selected_threshold": diverse_df.attrs.get("selected_threshold"),
    "warnings": diverse_df.attrs.get("warnings", []),
}

## 9. Show category summary

In [ ]:
pd.DataFrame(category_summary(diverse_df)).head(25)

## 10. Show sample items with images

In [ ]:
diverse_df.loc[
    diverse_df["primary_image_url"].notna(),
    ["parent_asin", "title", "main_category", "price_usd", "primary_image_url"],
].head(10)

## 11. Save CSVs and reports

In [ ]:
write_mvp_outputs(df, control_df, diverse_df, load_report, output_dir=ROOT / "analysis")
sorted(str(path.relative_to(ROOT)) for path in (ROOT / "analysis").glob("mvp_3000*"))

## 12. Final checks

In [ ]:
selected_threshold = diverse_df.attrs.get("selected_threshold", 150)
checks = {
    "row_count": len(diverse_df),
    "unique_parent_asin": diverse_df["parent_asin"].nunique(),
    "category_count": diverse_df["category_id"].nunique(),
    "avg_combined_words": diverse_df["combined_words"].mean(),
    "image_coverage": diverse_df["primary_image_url"].notna().mean(),
    "price_coverage": diverse_df["price_usd"].notna().mean(),
    "validation": validate_mvp_dataframe(diverse_df, min_combined_words=selected_threshold),
}
checks